In [10]:
# Setup Earth Engine initialization and define Larouco wildfire AOI

import ee
import geemap

# Connection to Google Earth Engine
ee.Authenticate()
ee.Initialize(project='wildfire-severity-mapping')

# Geometry definition for the area of interest (AOI)
center = ee.Geometry.Point([-7.0372, 42.5475])  # (longitude, latitude)
area_of_interest = center.buffer(25000)  # Buffer of 25 km around the point

# Map centered on the AOI creation
Map = geemap.Map(center=[42.5475, -7.0372], zoom=11) # (latitude, longitude)
Map.addLayer(area_of_interest, {'color': 'red'}, 'Area of Interest')

In [11]:
# Load and filter the Sentinel-2 image collection for the AOI and date range

# Call the Sentinel-2 image collection and filter it by the area of interest and cloud coverage
collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(area_of_interest)

# Function to mask clouds based on the SCL band
def cloud_mask(image):
    # Select the SCL band (cloud mask) and create a mask for clear pixels
    qa = image.select('SCL')
    cloud_mask = qa.neq(3).And(qa.neq(8)).And(qa.neq(9)).And(qa.neq(10))  # Keep only clear pixels

    return image.updateMask(cloud_mask)

# Date filtering for the wildfire event
pre_fire = collection.filterDate('2025-08-01', '2025-08-14').map(cloud_mask).median().clip(area_of_interest)  # Pre-fire period
post_fire = collection.filterDate('2025-09-01', '2025-09-15').map(cloud_mask).median().clip(area_of_interest)  # Post-fire period

In [12]:
# NVDi and NBR calculation for pre-fire and post-fire images

# Calculate NDVI for pre-fire image
pre_fire_ndvi = pre_fire.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Calculate NBR for pre-fire image
pre_fire_nbr = pre_fire.normalizedDifference(['B8', 'B12']).rename('NBR')

# Calculate NDVI for post-fire image
post_fire_ndvi = post_fire.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Calculate NBR for post-fire image
post_fire_nbr = post_fire.normalizedDifference(['B8', 'B12']).rename('NBR')


In [13]:
# Change detection during the wildfire event

dNDVI = pre_fire_ndvi.subtract(post_fire_ndvi).rename('dNDVI')

dNBR = pre_fire_nbr.subtract(post_fire_nbr).rename('dNBR')

In [14]:
# Map visualization parameters for NDVI and NBR

# Dicctionary for visualization parameters

ndvi_vis_params = {
    'min': -0.2,
    'max': 0.8,
    'palette': ['#1a9850', '#a6d96a', '#ffffff', '#f46d43', '#d73027']
}

nbr_vis_params = {
    'min': -0.1,
    'max': 0.8,
    'palette': ['#008000', '#ffff00', '#ff8c00', '#ff0000', '#800080']
}

# Add new map layers
Map.addLayer(dNDVI, ndvi_vis_params, 'Delta NDVI') # New map layer for dNDVI
Map.addLayer(dNBR, nbr_vis_params, 'Delta NBR') # New map layer for dNBR

Map 

Map(center=[42.5475, -7.0372], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [15]:
# Damage assessment based on dNBR values

affected_pixels = dNBR.gt(0.2)  # Threshold for affected area

# Calculate the area of affected pixels in square meters
affected_area = affected_pixels.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),  # Sum the pixel areas
    geometry=area_of_interest, # Use the AOI for the calculation
    scale=10,                  # Use a scale of 10 meters (Sentinel-2 resolution)
    maxPixels=1e9              # Maximum number of pixels to process
)

# Get the total area in square meters
square_meters = affected_area.get('dNBR').getInfo()

# Convert square meters to hectares (1 hectare = 10,000 square meters)
burned_hectares = square_meters / 10000

print(f"Estimated burned area: {burned_hectares:.2f} hectares")

Estimated burned area: 39504.37 hectares
